In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from datetime import datetime

# -------------------------------------------------------------
# CONFIGURACIÓN
# -------------------------------------------------------------
SECCIONES = {
    'portada':           'https://eljornalcr.com/',
    'realidad-local':    'https://eljornalcr.com/category/realidad-local/',
    'realidad-nacional': 'https://eljornalcr.com/category/realidad-nacional2/',
    'realidad-global':   'https://eljornalcr.com/category/realidad-global/',
    'deportes':          'https://eljornalcr.com/category/deportes/',
    'bienes-raices':     'https://eljornalcr.com/category/bienes-raices/',
    'jornal-literario':  'https://eljornalcr.com/category/el-jornal-literario/',
    'cultura':           'https://eljornalcr.com/category/cultura/',
    'opinion':           'https://eljornalcr.com/category/opinion/',
    'emprendimiento':    'https://eljornalcr.com/category/emprendedurismo/',
}

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
DELAY = 1.5

# -------------------------------------------------------------
# FUNCIONES
# -------------------------------------------------------------
def get_soup(url, session):
    try:
        response = session.get(url, timeout=10)
        response.raise_for_status()
        return BeautifulSoup(response.text, 'html.parser')
    except requests.RequestException as e:
        print(f"  Error: {url} -> {e}")
        return None

def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip() if text else ""

def extract_article_data(soup):
    # Título
    h1 = soup.find('h1')
    title = clean_text(h1.get_text()) if h1 else None

    # Fecha — "31 de octubre de 2025" → "2025-10-31"
    publication_date = None
    MESES = {'enero':'01','febrero':'02','marzo':'03','abril':'04','mayo':'05',
              'junio':'06','julio':'07','agosto':'08','septiembre':'09',
              'octubre':'10','noviembre':'11','diciembre':'12'}
    fecha_match = re.search(r'(\d{1,2})\s+de\s+(\w+)\s+de\s+(\d{4})', soup.get_text(' '))
    if fecha_match:
        dia, mes, anio = fecha_match.groups()
        mes_num = MESES.get(mes.lower())
        if mes_num:
            publication_date = f"{anio}-{mes_num}-{dia.zfill(2)}"

    # Sección
    section = None
    for a in soup.select('a'):
        if '/category/' in a.get('href', ''):
            section = clean_text(a.get_text()).lower()
            section = (section.replace('á','a').replace('é','e')
                               .replace('í','i').replace('ó','o')
                               .replace('ú','u').replace(' ','-'))
            break

    # Contenido
    full_text = None
    content_div = soup.select_one('.entry-content, .post-content, article')
    if content_div:
        for tag in content_div.find_all(['script', 'style', 'iframe', 'aside']):
            tag.decompose()
        full_text = clean_text(content_div.get_text())

    return title, publication_date, section, full_text

# -------------------------------------------------------------
# SCRAPING PRINCIPAL
# -------------------------------------------------------------
data = []
log_descartados = []
urls_vistas = set()

with requests.Session() as session:
    session.headers.update(HEADERS)

    for section, url in SECCIONES.items():
        print(f"\nSeccion: {section.upper()} -> {url}")
        main_soup = get_soup(url, session)
        if main_soup is None:
            print(f"  No se pudo acceder.")
            continue

        links = []
        for h3 in main_soup.find_all('h3'):
            a = h3.find('a', href=True)
            if a and 'eljornalcr.com' in a['href'] and a['href'] not in urls_vistas:
                links.append(a['href'])
                urls_vistas.add(a['href'])

        print(f"  {len(links)} articulos nuevos encontrados.")

        for i, link in enumerate(links, 1):
            print(f"    [{i}/{len(links)}] {link.split('/')[-2][:55]}...")
            time.sleep(DELAY)

            article_soup = get_soup(link, session)
            if article_soup:
                title, publication_date, section_real, full_text = extract_article_data(article_soup)
            else:
                title, publication_date, section_real, full_text = None, None, section, None

            scraping_date = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

            # Validaciones obligatorias
            if publication_date is None:
                log_descartados.append({'url': link, 'motivo': 'publication_date nulo'})
                continue
            if full_text is None or len(full_text) < 300:
                log_descartados.append({'url': link, 'motivo': f'full_text < 300 chars ({len(full_text) if full_text else 0})'})
                continue

            data.append({
                'source':           'eljornalcr',
                'url':              link,
                'title':            title,
                'publication_date': publication_date,
                'scraping_date':    scraping_date,
                'section':          section_real,
                'full_text':        full_text,
                'language':         'es'
            })

# -------------------------------------------------------------
# RESULTADO FINAL
# -------------------------------------------------------------
df = pd.DataFrame(data)
print(f"\nDataFrame: {len(df)} filas x {len(df.columns)} columnas")
print(f"Descartados: {len(log_descartados)}")
df.head(8)


Seccion: PORTADA -> https://eljornalcr.com/
  16 articulos nuevos encontrados.
    [1/16] la-ventana-de-los-libros-de-el-jornal...
    [2/16] centroamerica-futbol-en-transicion-permanente...
    [3/16] para-amantes-de-los-caballos...
    [4/16] el-aventurerismo-de-trump...
    [5/16] una-venus-de-pelicula...
    [6/16] la-falsa-america-de-donald-trump...
    [7/16] funeral-de-enrique-duran-sera-hoy-domingo-10-a-m-en-vue...
    [8/16] enrique-duran-una-vida-entregada-al-projimo...
    [9/16] enrique-duran-una-vida-de-servicio-al-projimo...
    [10/16] matonismo-contra-agricultores...
    [11/16] la-expropiacion-de-los-medios...
    [12/16] centroamerica-y-el-caribe-se-alinean-con-la-oms-en-la-l...
    [13/16] team-morita-el-canal-de-youtube-de-jose-eduardo-mora...
    [14/16] la-liga-se-viste-de-morado...
    [15/16] la-extrana-esta-estructura-en-el-techo-del-teatro-melic...
    [16/16] el-humor-inimitable-de-martin-amis...

Seccion: REALIDAD-LOCAL -> https://eljornalcr.com/category/re

,source,url,title,publication_date,scraping_date,section,full_text,language
0,eljornalcr,https://eljornalcr.com/la-ventana-de-los-libro...,La ventana de los libros de EL JORNAL,2026-03-30,2026-04-09 16:26:40,realidad-local,Inicio EL JORNAL LITERARIO La ventana de... EL...,es
1,eljornalcr,https://eljornalcr.com/centroamerica-futbol-en...,Centroamérica: fútbol en transición permanente,2026-03-27,2026-04-09 16:26:42,realidad-local,Inicio Deportes Centroamérica... DeportesReali...,es
2,eljornalcr,https://eljornalcr.com/para-amantes-de-los-cab...,Para amantes de los caballos,2026-02-22,2026-04-09 16:26:44,realidad-local,Inicio Deportes Para amantes ... Deportes Para...,es
3,eljornalcr,https://eljornalcr.com/el-aventurerismo-de-trump/,«El aventurerismo de Trump»,2026-01-03,2026-04-09 16:26:45,realidad-local,Inicio Inicio «El aventurer... InicioRealidad ...,es
4,eljornalcr,https://eljornalcr.com/una-venus-de-pelicula/,Una venus de película,2025-12-06,2026-04-09 16:26:47,realidad-local,Inicio Cultura Una venus de ... Cultura Una ve...,es
5,eljornalcr,https://eljornalcr.com/la-falsa-america-de-don...,La falsa «América» de Donald Trump,2025-10-31,2026-04-09 16:26:49,realidad-local,Inicio Realidad Local La falsa «Amé... Realida...,es
6,eljornalcr,https://eljornalcr.com/funeral-de-enrique-dura...,Funeral de Enrique Durán será hoy domingo 10 a...,2025-08-03,2026-04-09 16:26:50,realidad-local,Inicio Realidad Local Funeral de En... Realida...,es
7,eljornalcr,https://eljornalcr.com/enrique-duran-una-vida-...,Enrique Durán: una vida entregada al prójimo,2025-08-03,2026-04-09 16:26:52,realidad-local,Inicio Realidad Local Enrique Durán... Realida...,es


In [5]:
from google.colab import files
df.to_csv('eljornalcr.csv', index=False, encoding='utf-8', sep=',', na_rep='NULL')
files.download('eljornalcr.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>